In [0]:
%pip install great-expectations==0.18.21


  Using cached great_expectations-0.18.21-py3-none-any.whl.metadata (8.5 kB)
  Using cached altair-4.2.2-py3-none-any.whl.metadata (13 kB)
  Using cached colorama-0.4.6-py2.py3-none-any.whl.metadata (17 kB)
  Using cached makefun-1.16.0-py2.py3-none-any.whl.metadata (2.9 kB)
  Using cached ruamel.yaml-0.17.40-py3-none-any.whl.metadata (19 kB)
  Using cached tzlocal-5.3.1-py3-none-any.whl.metadata (7.6 kB)
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached entrypoints-0.4-py3-none-any.whl.metadata (2.6 kB)
  Using cached toolz-1.1.0-py3-none-any.whl.metadata (5.1 kB)
  Using cached ruamel_yaml_clib-0.2.15-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (3.5 kB)
Using cached great_expectations-0.18.21-py3-none-any.whl (5.4 MB)
Using cached altair-4.2.2-py3-none-any.whl (813 kB)
Using cached colorama-0.4.6-py2.py3-none-any.whl (25 kB)
Using cached makefun-1.16.0-py2.py3-none-any.w

# Gold GE Audit

Audit the final published Gold layer and monitoring tables, then publish Gold Data Docs.


In [0]:
from datetime import datetime, timezone
import great_expectations as gx
from great_expectations.checkpoint import Checkpoint
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql import types as T
import uuid

In [0]:
STORAGE_ACCOUNT = "hantstorageaccount"
LAKEHOUSE_CONTAINER = "lakehouse"

CATALOG_NAME = "hant-catalog"
SCHEMA_NAME = "invoice"

GOLD_HEADER_PATH = dbutils.jobs.taskValues.get(
    taskKey="gold_publish",
    key="gold_header_path",
    debugValue=f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/gold/validated/header/",
)
GOLD_LINES_PATH = dbutils.jobs.taskValues.get(
    taskKey="gold_publish",
    key="gold_lines_path",
    debugValue=f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/gold/validated/lines/",
)
GOLD_ISSUE_LOG_PATH = dbutils.jobs.taskValues.get(
    taskKey="gold_publish",
    key="gold_issue_log_path",
    debugValue=f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/gold/ge/issue_log/",
)
GOLD_RUN_SUMMARY_PATH = dbutils.jobs.taskValues.get(
    taskKey="gold_publish",
    key="gold_run_summary_path",
    debugValue=f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/gold/monitoring/run_summary/",
)
GOLD_SOURCE_SUMMARY_PATH = dbutils.jobs.taskValues.get(
    taskKey="gold_publish",
    key="gold_source_publication_summary_path",
    debugValue=f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/gold/monitoring/source_publication_summary/",
)
GOLD_DQ_IMPACT_PATH = dbutils.jobs.taskValues.get(
    taskKey="gold_publish",
    key="gold_dq_impact_summary_path",
    debugValue=f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/gold/monitoring/dq_impact_summary/",
)
GOLD_KPI_RECON_PATH = dbutils.jobs.taskValues.get(
    taskKey="gold_publish",
    key="gold_kpi_recon_summary_path",
    debugValue=f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/gold/monitoring/kpi_reconciliation_summary/",
)

GX_ROOT = "/dbfs/tmp/ge/gold"
RUN_ID = str(uuid.uuid4())
SITE = "gold_local_site"
PIPELINE_LAYER = "gold"
FILESTORE_DATA_DOCS_DBFS_PATH = "dbfs:/FileStore/great_expectations/gold/"
FILESTORE_DATA_DOCS_DBFS_FUSE_PATH = "/dbfs/FileStore/great_expectations/gold"
FILESTORE_DATA_DOCS_BROWSER_PATH = "/files/great_expectations/gold/index.html"

STORAGE_DATA_DOCS_LATEST_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/monitoring/data_docs/gold/latest/"
STORAGE_DATA_DOCS_RUNS_ROOT = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/monitoring/data_docs/gold/runs/"

GE_RUNTIME_METRICS_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/monitoring/ge_runtime_metrics/"
GE_RUNTIME_METRICS_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.batch_ge_runtime_metrics"

ALLOWED_SOURCE_TYPES = ["csv", "json"]
ALLOWED_SHIP_MODES = [
    "FIRST CLASS",
    "SECOND CLASS",
    "STANDARD CLASS",
    "SAME DAY",
]
ALLOWED_IMPACTS = [
    "warning_header",
    "warning_lines",
    "error_header",
    "error_lines",
]

GE_RUNTIME_SCHEMA = T.StructType([
    T.StructField("layer", T.StringType(), False),
    T.StructField("suite_name", T.StringType(), False),
    T.StructField("run_id", T.StringType(), True),
    T.StructField("started_at", T.TimestampType(), False),
    T.StructField("ended_at", T.TimestampType(), False),
    T.StructField("runtime_seconds", T.DoubleType(), False),
    T.StructField("rows_evaluated", T.LongType(), True),
    T.StructField("rows_per_second", T.DoubleType(), True),
    T.StructField("evaluated_expectations", T.LongType(), True),
    T.StructField("successful_expectations", T.LongType(), True),
    T.StructField("failed_expectations", T.LongType(), True),
    T.StructField("expectation_success_percent", T.DoubleType(), True),
    T.StructField("validation_success", T.BooleanType(), True),
    T.StructField("recorded_at", T.TimestampType(), False),
])

def ctx():
    return gx.get_context(context_root_dir=GX_ROOT)


def ds(c, name):
    try:
        return c.sources.add_or_update_spark(name=name)
    except AttributeError:
        c.add_datasource(
            name,
            class_name="Datasource",
            execution_engine={"class_name": "SparkDFExecutionEngine"},
            data_connectors={
                "runtime_data_connector": {
                    "class_name": "RuntimeDataConnector",
                    "batch_identifiers": ["default_identifier_name"],
                }
            },
        )
        return c.get_datasource(name)


def req(d, n, df):
    return d.add_dataframe_asset(name=n).build_batch_request(dataframe=df)


def ck(c, n, r, s):
    x = Checkpoint(
        name=n,
        data_context=c,
        validations=[{"batch_request": r, "expectation_suite_name": s}],
        action_list=[
            {
                "name": "store_validation_result",
                "action": {"class_name": "StoreValidationResultAction"},
            },
            {
                "name": "update_data_docs",
                "action": {"class_name": "UpdateDataDocsAction"},
            },
        ],
        run_name_template="%Y%m%dT%H%M%S_gold_ge",
    )
    c.add_or_update_checkpoint(checkpoint=x)
    return x


def first(x):
    return x.run_results[list(x.run_results.keys())[0]]["validation_result"]


def summ(v, s):
    st = v.get("statistics", {}) or {}
    return {
        "suite": s,
        "success": v.get("success"),
        "evaluated": st.get("evaluated_expectations", 0),
        "successful": st.get("successful_expectations", 0),
        "failed": st.get("unsuccessful_expectations", 0),
    }


def show(s):
    print("=" * 60)
    print(f"{s['suite']}: {'PASSED' if s['success'] else 'FAILED'}")
    print(
        f"Evaluated={s['evaluated']} Passed={s['successful']} Failed={s['failed']}"
    )

def ensure_data_docs_site(context):
    try:
        context.delete_data_docs_site(SITE)
    except Exception:
        pass

    context.add_data_docs_site(
        site_name=SITE,
        site_config={
            "class_name": "SiteBuilder",
            "store_backend": {
                "class_name": "TupleFilesystemStoreBackend",
                "base_directory": FILESTORE_DATA_DOCS_DBFS_FUSE_PATH,
            },
            "site_index_builder": {"class_name": "DefaultSiteIndexBuilder"},
        },
    )

def _safe_rm(path: str):
    try:
        dbutils.fs.rm(path, True)
    except Exception:
        pass


def _safe_ls(path: str, label: str):
    print(f"\n{label}: {path}")
    try:
        for x in dbutils.fs.ls(path):
            print(f" - {x.path}")
    except Exception as e:
        print(f"   [unable to list] {e}")


def publish_data_docs(context, layer_name=PIPELINE_LAYER):
    # Clean FileStore target first so stale HTML does not remain
    _safe_rm(FILESTORE_DATA_DOCS_DBFS_PATH)

    # Build the site directly into FileStore
    context.build_data_docs(site_names=[SITE])

    # Verify the expected folders exist
    _safe_ls(FILESTORE_DATA_DOCS_DBFS_PATH, "Data Docs root")
    _safe_ls(FILESTORE_DATA_DOCS_DBFS_PATH + "expectations/", "Data Docs expectations")
    _safe_ls(FILESTORE_DATA_DOCS_DBFS_PATH + "validations/", "Data Docs validations")

    run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    storage_run_path = f"{STORAGE_DATA_DOCS_RUNS_ROOT}{run_stamp}/"

    # Refresh ADLS copies from FileStore
    _safe_rm(STORAGE_DATA_DOCS_LATEST_PATH)
    dbutils.fs.cp(FILESTORE_DATA_DOCS_DBFS_PATH, STORAGE_DATA_DOCS_LATEST_PATH, True)
    dbutils.fs.cp(FILESTORE_DATA_DOCS_DBFS_PATH, storage_run_path, True)

    return {
        "preview_url": FILESTORE_DATA_DOCS_BROWSER_PATH,
        "storage_latest_path": STORAGE_DATA_DOCS_LATEST_PATH,
        "storage_run_path": storage_run_path,
    }


def render_data_docs_preview(data_docs_info, title):
    displayHTML(
        f'''
        <div style="margin:16px 0;">
          <p><strong>{title}</strong></p>
          <p><a href="{data_docs_info["preview_url"]}" target="_blank">Open Data Docs index</a></p>
          <iframe
            src="{data_docs_info["preview_url"]}"
            width="100%"
            height="900"
            style="border:1px solid #d0d7de;border-radius:8px;background:#fff;">
          </iframe>
        </div>
        '''
    )


In [0]:
# runtime metrics helper functions
def first_validation_result(checkpoint_result):
    run_results = checkpoint_result.run_results
    return run_results[list(run_results.keys())[0]]["validation_result"]


def validation_statistics(validation_result):
    stats = validation_result.get("statistics", {}) or {}
    evaluated = int(stats.get("evaluated_expectations", 0) or 0)
    successful = int(stats.get("successful_expectations", 0) or 0)
    failed = int(stats.get("unsuccessful_expectations", 0) or 0)
    success_percent = float(successful / evaluated) if evaluated else None
    return evaluated, successful, failed, success_percent


def write_ge_runtime_metric(layer, suite_name, run_id, started_at, ended_at, rows_evaluated, validation_result):
    runtime_seconds = (ended_at - started_at).total_seconds()
    evaluated, successful, failed, success_percent = validation_statistics(validation_result)
    rows_per_second = float(rows_evaluated / runtime_seconds) if rows_evaluated and runtime_seconds > 0 else None

    row = [(
        layer,
        suite_name,
        run_id,
        started_at,
        ended_at,
        float(runtime_seconds),
        int(rows_evaluated) if rows_evaluated is not None else None,
        rows_per_second,
        evaluated,
        successful,
        failed,
        success_percent,
        bool(validation_result.get("success")),
        datetime.now(timezone.utc),
    )]

    runtime_df = spark.createDataFrame(row, GE_RUNTIME_SCHEMA)
    runtime_df.write.format("delta").mode("append").save(GE_RUNTIME_METRICS_PATH)

    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG_NAME}`.{SCHEMA_NAME}")
    spark.sql(f'''
        CREATE TABLE IF NOT EXISTS {GE_RUNTIME_METRICS_TABLE}
        USING DELTA
        LOCATION "{GE_RUNTIME_METRICS_PATH}"
    ''')


In [0]:
h = spark.read.format("delta").load(GOLD_HEADER_PATH)
l = spark.read.format("delta").load(GOLD_LINES_PATH)
issues = spark.read.format("delta").load(GOLD_ISSUE_LOG_PATH)
run = spark.read.format("delta").load(GOLD_RUN_SUMMARY_PATH)
src = spark.read.format("delta").load(GOLD_SOURCE_SUMMARY_PATH)
impact = spark.read.format("delta").load(GOLD_DQ_IMPACT_PATH)
kpi = spark.read.format("delta").load(GOLD_KPI_RECON_PATH)

if h.rdd.isEmpty() or l.rdd.isEmpty():
    raise RuntimeError("Gold GE failure: validated Gold tables are empty.")

if (
    h.groupBy("InvoiceId").count().filter(col("count") > 1).count() > 0
    or l.groupBy("InvoiceId", "LineNumber").count().filter(col("count") > 1).count() > 0
    or l.join(h.select("InvoiceId").distinct(), on="InvoiceId", how="left_anti").count() > 0
    or h.join(l.select("InvoiceId").distinct(), on="InvoiceId", how="left_anti").count() > 0
):
    raise RuntimeError("Gold GE failure: final serving keys are inconsistent.")

c = ctx()
d = ds(c, "gold_runtime")


def run_suite(df, asset, suite, build):
    r = req(d, asset, df)
    c.add_or_update_expectation_suite(expectation_suite_name=suite)
    v = c.get_validator(batch_request=r, expectation_suite_name=suite)
    build(v)
    v.save_expectation_suite(discard_failed_expectations=False)
    ck(c, f"{suite}_ck", r, suite)
    started_at = datetime.now(timezone.utc)
    checkpoint_result = c.run_checkpoint(checkpoint_name=f"{suite}_ck")
    ended_at = datetime.now(timezone.utc)
    validation_result = first_validation_result(checkpoint_result)
    write_ge_runtime_metric(
        layer="gold",
        suite_name=suite,
        run_id=None,
        started_at=started_at,
        ended_at=ended_at,
        rows_evaluated=df.count(),
        validation_result=validation_result,
    )
    return checkpoint_result


res = [
    run_suite(
        h,
        "gold_header_audit_asset",
        "gold_header_audit_suite",
        lambda v: [
            v.expect_column_values_to_not_be_null("InvoiceId"),
            v.expect_column_values_to_be_unique("InvoiceId"),
            v.expect_column_values_to_be_in_set("SourceType", ALLOWED_SOURCE_TYPES),
            v.expect_column_values_to_be_in_set("ShipMode", ALLOWED_SHIP_MODES),
            v.expect_column_values_to_be_between("InvoiceTotal", min_value=0),
        ],
    ),
    run_suite(
        l,
        "gold_lines_audit_asset",
        "gold_lines_audit_suite",
        lambda v: [
            v.expect_column_values_to_not_be_null("InvoiceId"),
            v.expect_column_values_to_not_be_null("LineNumber"),
            v.expect_compound_columns_to_be_unique(["InvoiceId", "LineNumber"]),
            v.expect_column_values_to_be_between("Quantity", min_value=1),
            v.expect_column_values_to_be_between("UnitPrice", min_value=0),
            v.expect_column_values_to_be_between("ItemSubTotal", min_value=0),
        ],
    ),
    run_suite(
        run,
        "gold_run_audit_asset",
        "gold_run_audit_suite",
        lambda v: [
            v.expect_column_values_to_be_between("PublishedInvoiceCount", min_value=1),
            v.expect_column_values_to_be_between("PublishedLineCount", min_value=1),
            v.expect_column_values_to_be_between("HeaderDuplicateCount", min_value=0, max_value=0),
            v.expect_column_values_to_be_between("LineDuplicateCount", min_value=0, max_value=0),
            v.expect_column_values_to_be_between("OrphanLineCount", min_value=0, max_value=0),
            v.expect_column_values_to_be_between("HeaderWithoutLinesCount", min_value=0, max_value=0),
        ],
    ),
    run_suite(
        src,
        "gold_source_audit_asset",
        "gold_source_audit_suite",
        lambda v: [
            v.expect_column_values_to_be_in_set("SourceType", ALLOWED_SOURCE_TYPES),
            v.expect_column_values_to_be_between("PublishedInvoiceCount", min_value=0),
            v.expect_column_values_to_be_between("PublishedRevenue", min_value=0),
        ],
    ),
    run_suite(
        impact,
        "gold_impact_audit_asset",
        "gold_impact_audit_suite",
        lambda v: [
            v.expect_column_values_to_be_in_set("ImpactType", ALLOWED_IMPACTS),
            v.expect_column_values_to_be_between("AffectedInvoices", min_value=0),
            v.expect_column_values_to_be_between("AffectedLines", min_value=0),
            v.expect_column_values_to_be_between("AffectedRules", min_value=0),
        ],
    ),
    run_suite(
        kpi,
        "gold_kpi_audit_asset",
        "gold_kpi_audit_suite",
        lambda v: [
            v.expect_column_values_to_not_be_null("InvoiceId"),
            v.expect_column_values_to_be_between("PublishedLineCount", min_value=1),
            v.expect_column_values_to_be_between("HeaderVsLinesSubTotalDiff", min_value=-0.05, max_value=0.05),
            v.expect_column_values_to_be_between("HeaderVsLinesTotalDiff", min_value=-0.05, max_value=0.05),
        ],
    ),
    run_suite(
        issues,
        "gold_issue_audit_asset",
        "gold_issue_audit_suite",
        lambda v: [
            v.expect_column_values_to_not_be_null("rule_id"),
            v.expect_column_values_to_be_in_set("severity", ["WARNING", "ERROR"]),
            v.expect_column_values_to_not_be_null("dq_reason"),
            v.expect_column_values_to_not_be_null("GoldRunId"),
        ],
    ),
]

for s in [
    summ(first(x), n)
    for x, n in zip(
        res,
        [
            "gold_header_audit_suite",
            "gold_lines_audit_suite",
            "gold_run_audit_suite",
            "gold_source_audit_suite",
            "gold_impact_audit_suite",
            "gold_kpi_audit_suite",
            "gold_issue_audit_suite",
        ],
    )
]:
    show(s)

if any(not first(x).get("success") for x in res):
    raise RuntimeError(
        "Gold GE validation failed. Review the Gold Data Docs for details."
    )

# Build and publish Gold Data Docs after all checkpoints have run
ensure_data_docs_site(c)

gold_data_docs = publish_data_docs(c)
render_data_docs_preview(gold_data_docs, "Gold Great Expectations Data Docs")

    - No action was taken.
  warnings.warn(message)

    - No action was taken.
  warnings.warn(message)

INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmpn_fb4d4t' for ephemeral docs site
  warnings.warn(



Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/11 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/11 [00:00<?, ?it/s]

...

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/31 [00:00<?, ?it/s]

gold_header_audit_suite: PASSED
Evaluated=5 Passed=5 Failed=0
gold_lines_audit_suite: PASSED
Evaluated=6 Passed=6 Failed=0
gold_run_audit_suite: PASSED
Evaluated=6 Passed=6 Failed=0
gold_source_audit_suite: PASSED
Evaluated=3 Passed=3 Failed=0
gold_impact_audit_suite: PASSED
Evaluated=4 Passed=4 Failed=0
gold_kpi_audit_suite: PASSED
Evaluated=4 Passed=4 Failed=0
gold_issue_audit_suite: PASSED
Evaluated=4 Passed=4 Failed=0

Data Docs root: dbfs:/FileStore/great_expectations/gold/
 - dbfs:/FileStore/great_expectations/gold/expectations/
 - dbfs:/FileStore/great_expectations/gold/index.html
 - dbfs:/FileStore/great_expectations/gold/static/
 - dbfs:/FileStore/great_expectations/gold/validations/

Data Docs expectations: dbfs:/FileStore/great_expectations/gold/expectations/
 - dbfs:/FileStore/great_expectations/gold/expectations/gold_header_audit_suite.html
 - dbfs:/FileStore/great_expectations/gold/expectations/gold_impact_audit_suite.html
 - dbfs:/FileStore/great_expectations/gold/expect

Gold Great Expectations Data Docs 
 Open Data Docs index

In [0]:
# Databricks table registration for Metaplane: Gold GE monitored outputs
CATALOG_NAME = "hant-catalog"
SCHEMA_NAME = "invoice"

BATCH_GOLD_VALIDATED_HEADER_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.batch_gold_validated_header"
BATCH_GOLD_VALIDATED_LINES_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.batch_gold_validated_lines"
BATCH_GOLD_GE_ISSUE_LOG_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.batch_gold_ge_issue_log"
BATCH_GOLD_RUN_SUMMARY_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.batch_gold_run_summary"
BATCH_GOLD_SOURCE_SUMMARY_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.batch_gold_source_publication_summary"
BATCH_GOLD_DQ_IMPACT_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.batch_gold_dq_impact_summary"
BATCH_GOLD_KPI_RECON_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.batch_gold_kpi_reconciliation_summary"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG_NAME}`.{SCHEMA_NAME}")

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {BATCH_GOLD_VALIDATED_HEADER_TABLE}
USING DELTA
LOCATION "{GOLD_HEADER_PATH}"
''')

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {BATCH_GOLD_VALIDATED_LINES_TABLE}
USING DELTA
LOCATION "{GOLD_LINES_PATH}"
''')

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {BATCH_GOLD_GE_ISSUE_LOG_TABLE}
USING DELTA
LOCATION "{GOLD_ISSUE_LOG_PATH}"
''')

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {BATCH_GOLD_RUN_SUMMARY_TABLE}
USING DELTA
LOCATION "{GOLD_RUN_SUMMARY_PATH}"
''')

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {BATCH_GOLD_SOURCE_SUMMARY_TABLE}
USING DELTA
LOCATION "{GOLD_SOURCE_SUMMARY_PATH}"
''')

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {BATCH_GOLD_DQ_IMPACT_TABLE}
USING DELTA
LOCATION "{GOLD_DQ_IMPACT_PATH}"
''')

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {BATCH_GOLD_KPI_RECON_TABLE}
USING DELTA
LOCATION "{GOLD_KPI_RECON_PATH}"
''')

display(spark.sql(f"DESCRIBE DETAIL {BATCH_GOLD_RUN_SUMMARY_TABLE}"))


format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,35b38f66-15eb-485a-a07c-9859c9f81151,hant-catalog.invoice.batch_gold_run_summary,null,abfss://lakehouse@hantstorageaccount.dfs.core.windows.net/gold/monitoring/run_summary,2026-04-23T19:53:12.521Z,2026-04-23T19:53:13Z,List(),List(),1,6042,Map(delta.enableDeletionVectors -> true),3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false
